# A/B cross-corpus, hai chiều, **7 lớp chung**Bốn lần train. Trong mỗi cặp, hai nhánh khác nhau đúng **một** biến: `--use-uot`.| Chiều | Train | Đánh giá ||---|---|---|| `dfew2mafw` | DFEW (7 lớp) | MAFW-7 || `mafw2dfew` | **MAFW-7** (đã bỏ 4 lớp riêng) | DFEW (7 lớp) |**Mọi thí nghiệm đều trong không gian 7 lớp chung.** MAFW mặc định 11 lớp nên bị ép về 7bằng `--num-classes 7` cùng split `_faces7`. DFEW 7 lớp là 7 lớp đầu của MAFW **cùng thứtự**, nên không cần ánh xạ nhãn.Checkpoint MAFW của tác giả có head 11 lớp; khi `--resume` vào model 7 lớp, `main.py` tự**cắt 7 hàng đầu** của head và in ra dòng `head sliced to the shared label space`.Chọn `DIRECTION` ở Cell 2, chạy hết, rồi đổi chiều và chạy lại. Mỗi chiều ~8 giờ.UOT đụng vào đâu trong pipeline gốc: `UOT_DIFF.md`.

## 1. Clone

In [ ]:
WORK = "/kaggle/working/DEFR-UOT"
import os, shutil
if os.path.exists(WORK):
    shutil.rmtree(WORK)
!git clone -b feat/uot-fusion --depth 1 https://github.com/YouttyLe-DSAI/DEFR-UOT.git {WORK}
%cd {WORK}
!git log --oneline -1

## 2. CONFIG — cell duy nhất cần sửa

In [ ]:
# Chon MOT chieu, chay het notebook, roi doi sang chieu con lai va chay lai.DIRECTION = "dfew2mafw"        # "dfew2mafw"  hoac  "mafw2dfew"DATA      = "/kaggle/temp/data"MODEL_DIR = "/kaggle/input/models/tunalmt/modelmma/pytorch/default/1"CKPT      = f"{MODEL_DIR}/checkpoint"EPOCHS, BATCH, LR, WORKERS, FOLD = 5, 4, 2e-5, 2, 1# CA HAI chieu deu chay trong khong gian 7 lop chung.# MAFW mac dinh 11 lop -> ep ve 7 va dung split _faces7.N_CLASSES = 7if DIRECTION == "dfew2mafw":    TRAIN_ON  = "DFEW"    TRAIN_ANN = ""                                               # DFEW von da 7 lop    TEST_ANN  = "./annotation/MAFW_set_{fold}_test_faces7.txt"else:    TRAIN_ON  = "MAFW"    TRAIN_ANN = "./annotation/MAFW_set_{fold}_train_faces7.txt"   # bo 4 lop rieng    TEST_ANN  = "./annotation/DFEW_set_{fold}_test.txt"RESUME = f"{CKPT}/{TRAIN_ON}_224/fold{{fold}}_224.pth"ARGS = f'--num-classes {N_CLASSES} --test-annotation "{TEST_ANN}"'if TRAIN_ANN:    ARGS += f' --train-annotation "{TRAIN_ANN}"'print(f"{DIRECTION}: train {TRAIN_ON} ({N_CLASSES} lop)  ->  danh gia {TEST_ANN}")print("resume:", RESUME)print("args  :", ARGS)

## 3. Dependencies

`timm==0.9.16` bắt buộc — image Kaggle dùng timm 1.x, mà `models/models_vit.py` kế thừa
`timm.models.vision_transformer.VisionTransformer`, API đổi giữa hai major version.
Không cài đè torch (mất bản CUDA của Kaggle).

In [ ]:
!pip install -q timm==0.9.16 einops==0.7.0 librosa==0.10.1
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 4. Dựng dữ liệu

Dataloader suy ra đường dẫn `.wav` **từ đường dẫn frame bằng phép thay chuỗi**, mà trên
Kaggle các shard nằm ở những mount khác nhau. Cell này dựng cây symlink đúng layout
loader mong đợi — không sửa một dòng nào của code gốc.

Kiểm tra output: `LAYOUT OK`, và `clips w/o wav` phải nhỏ (MAFW) / bằng 0 (DFEW).

In [ ]:
# Cross-corpus nen can CA HAI corpus: DFEW de train, MAFW de danh gia.
for ds, root in (("DFEW", f"{DATA}/dfew/clip_224x224"), ("MAFW", f"{DATA}/mfaw/clips_faces")):
    print(f"\n{'='*60}\n{ds}\n{'='*60}")
    !python tools/kaggle_setup.py --dataset {ds} --auto --out {DATA}
    !python tools/retarget_annotations.py --dataset {ds} --new-root {root} --recount --drop-missing

## 5. Trỏ annotation

Điều kiện đi tiếp: `missing folder 0`, `EMPTY folder 0`, **`UNMATCHED path 0`**,
`labels seen` = `0..10` (MAFW) / `0..6` (DFEW).

In [ ]:
# MAFW-7 cho ca train lan test: bo cac dong co nhan >= 7.!python tools/make_mafw7.py --split bothTEST_FILE = TEST_ANN.format(fold=FOLD)!python tools/check_data.py --annotation {TEST_FILE} --n 300

## 6. Checkpoint encoder

Hai file này chỉ dùng lúc **dựng** model; `--resume` sẽ ghi đè trọng số ngay sau đó.
Nhưng thiếu chúng thì `GenerateModel.__init__` chết ở `torch.load`.

In [ ]:
!cp {CKPT}/pretrained.pth ./audiomae_pretrained.pth
!cp {CKPT}/mae_face_visualize_vit_base.pth ./mae_face_pretrain_vit_base.pth
!ls -la *.pth

## 7. Smoke test — 1 phút, đừng bỏ qua

Cần thấy:

- **`audio liveness ... 0/N dead`** — `N/N dead` nghĩa là các `.wav` không được tìm thấy
  và audio đang là hằng số. Dừng lại.
- **`max|baseline - uot| = 0.000e+00  OK`** — dựng model có UOT rồi tắt cờ thì output
  trùng khít bản gốc. Đây là bằng chứng UOT không phá pipeline gốc.
- gradient của **gate** khác 0 sau vài bước (`proj`/`norm` bằng 0 ở step 0 là đúng:
  chúng nằm sau `tanh(gate)=0`)
- `peak GPU memory` — cho biết `BATCH` nào vừa VRAM

In [ ]:
# Smoke test phai dung DUNG cau hinh cua thi nghiem: 7 lop, va dung split MAFW-7# neu dang train tren MAFW. Sai o day thi so lieu ve sau khong so duoc.SMOKE = f"--num-classes {N_CLASSES}"if TRAIN_ANN:    SMOKE += f' --train-annotation "{TRAIN_ANN}"'!python tools/smoke_test.py --dataset {TRAIN_ON} {SMOKE} --use-uot --batch-size 2

## 8. NHÁNH A — pipeline gốc

Không có `--use-uot`. Đây là MMA-DFER nguyên bản.

**Soi ngay khi chạy:**

```
Resumed from .../fold1_224.pth
  missing (OTHER -- should be 0): 0      <- khác 0 = checkpoint không khớp
Epoch: [0][0/1833]  Loss 0.70xx  Accuracy 75.000
```

**Loss step 0 phải ~0.7.** Ra ~2.9 nghĩa là warm-start không ăn (classifier đang ngẫu
nhiên) — dừng ngay, đừng để chạy 4 giờ.

Thời gian: ~45–50 phút/epoch → **~4 giờ**.

In [ ]:
!python main.py --dataset {TRAIN_ON} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --resume "{RESUME}" {ARGS} \  --exper-name AB_BASE_{DIRECTION}

## 9. NHÁNH B — pipeline + UOT

Khác nhánh A **đúng một thứ**: `--use-uot` và các tham số của nó.

Vì gate khởi tạo bằng 0, cell này **phải bắt đầu từ đúng loss như nhánh A**. Khác đi là
dấu hiệu UOT phá gì đó ngay lúc khởi tạo — kiểm tra miễn phí, tận dụng đi.

In [ ]:
!python main.py --dataset {TRAIN_ON} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --resume "{RESUME}" {ARGS} \  --use-uot --uot-eps 0.05 --uot-tau 1.0 --uot-iters 10 \  --exper-name AB_UOT_{DIRECTION}

## 10. So sánh

Đọc thẳng từ `log.txt` của hai run. Dòng `Confusion Matrix Diag` mà `main.py` ghi ra
chính là **recall từng lớp** (đường chéo ma trận nhầm lẫn đã chuẩn hoá theo hàng), và
UAR là trung bình của nó — nên bảng per-class đã có sẵn, chỉ cần đọc.

In [ ]:
!python tools/compare_runs.py --log-root log --base AB_BASE_{DIRECTION} --uot AB_UOT_{DIRECTION}

## Ghi chú

- Hai nhánh ~8 giờ. Vừa một phiên 12 giờ, nhưng **an toàn hơn là mỗi nhánh một phiên**
  (mỗi phiên có 12 giờ riêng). Dùng **Save Version → Save & Run All (Commit)**; phiên
  tương tác bị ngắt khi idle.
- `/kaggle/temp` bị xoá sau mỗi phiên → chạy lại Cell 4 mỗi lần mở notebook.
- Kết quả này trả lời *"UOT có cải thiện thêm trên một model đã hội tụ không"*. Muốn
  khẳng định UOT tốt hơn **khi train từ đầu** thì phải bỏ `--resume`, chạy đủ 25 epoch
  × 5 fold — việc đó cần server, Kaggle không đủ.